# NLP II — Generative Model Lab (Colab) — **Student Template**

🧪 **Theme:** Homebrewing beer (Spanish literals kept intentionally).

You will complete missing pieces marked with **TODO**. All instructions and comments are in English, while sample **text literals remain in Spanish**. The goal is to:
1. Load a small open LLM in Colab Free (GPU or CPU).
2. Implement a **chat generation function** (`generate_chat`).
3. Implement a **paraphrasing/style** function (`restyle`).
4. Implement **sentiment classification via prompting** (`classify_sentiment`) and evaluate **zero-shot vs few-shot**.
5. Explore **decoding parameters**.
6. Build a simple **multi-turn conversation** loop.
7. Use a mini **BJCP style table** to craft recipe prompts.

▶️ **Run order**: Install → Load model → Implement `generate_chat` → Playground → Paraphrase → Sentiment → Decoding → Multi-turn → BJCP.

💡 **Note**: If `temperature == 0`, use **greedy** (`do_sample=False`).

## 1) Install dependencies
Run once. It may take ~1–2 minutes.

In [ ]:
%%bash
pip -q install --upgrade transformers accelerate sentencepiece
pip -q install bitsandbytes scikit-learn matplotlib pandas


## 2) Load model (auto-select)
We auto-detect GPU. If available, try `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit; otherwise fallback to `TinyLlama/TinyLlama-1.1B-Chat-v1.0` on CPU.

You don't need to change this cell.

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

PREFERRED_GPU_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
FALLBACK_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

def safe_load(model_name, fourbit=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    kwargs = {}
    if fourbit and DEVICE=='cuda':
        try:
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
            kwargs.update(dict(quantization_config=bnb, device_map='auto', torch_dtype=torch.float16))
        except Exception as e:
            print('bitsandbytes not available, loading without 4-bit:', e)
            kwargs.update(dict(device_map='auto', torch_dtype=torch.float16))
    else:
        if DEVICE == 'cuda':
            kwargs.update(dict(device_map='auto', torch_dtype=torch.float16))
        else:
            kwargs.update(dict(device_map={'': 'cpu'}, torch_dtype=torch.float32))
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.eval()
    return tokenizer, model

try:
    if DEVICE == 'cuda':
        print('Trying 4-bit:', PREFERRED_GPU_MODEL)
        tokenizer, model = safe_load(PREFERRED_GPU_MODEL, fourbit=True)
        SELECTED = PREFERRED_GPU_MODEL
    else:
        print('CPU mode:', FALLBACK_MODEL)
        tokenizer, model = safe_load(FALLBACK_MODEL, fourbit=False)
        SELECTED = FALLBACK_MODEL
except Exception as e:
    print('Preferred model failed:', e)
    print('Fallback to CPU:', FALLBACK_MODEL)
    if DEVICE=='cuda': torch.cuda.empty_cache()
    gc.collect()
    tokenizer, model = safe_load(FALLBACK_MODEL, fourbit=False)
    SELECTED = FALLBACK_MODEL

print('Selected model =>', SELECTED)
EOS = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.convert_tokens_to_ids('</s>')

## 3) `generate_chat` — **Implement this**
Create a function that:
- Uses `tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` to build a prompt.
- Tokenizes and **sends to the same device** as the model.
- Calls `model.generate(...)` with the given decoding params.
- Decodes and returns only the **generated continuation** (skip the prompt).
- 🔴 **Edge-case**: if `do_sample=True` but `temperature<=0`, **switch to greedy** (`do_sample=False`). In greedy, temperature is ignored.

**Signature:**
```python
def generate_chat(messages, temperature=0.2, top_p=0.95, max_new_tokens=256, do_sample=True):
    ...
```
Return: `str` (the assistant's response).

In [ ]:
# TODO: implement generate_chat as described above.
def generate_chat(messages, temperature=0.2, top_p=0.95, max_new_tokens=256, do_sample=True):
    """Implement chat generation using the model and tokenizer.
    - messages: list of dicts with roles {system,user,assistant} and 'content' strings (Spanish literals allowed).
    - handle temperature==0 by switching to greedy (do_sample=False).
    - decode only new tokens (exclude the prompt), and strip spaces.
    """
    raise NotImplementedError("Complete generate_chat() as per the instructions in the previous cell.")

## Playground — Prompting (edit and run)
After implementing `generate_chat`, try your own prompts. **Keep literals in Spanish**, comments in English.

In [ ]:
# Edit messages & parameters below (Spanish literals are encouraged):
messages = [
    {"role": "system", "content": "Eres un experto en elaboración de cerveza casera. Responde en español, con precisión y sin inventar."},
    {"role": "user",   "content": "¿Qué diferencia hay entre una IPA americana y una German Pils en aroma y amargor?"}
]
temperature = 0.2
top_p = 0.95
do_sample = True  # set False for greedy
max_new_tokens = 160

# Uncomment after implementing generate_chat()
# print(generate_chat(messages, temperature=temperature, top_p=top_p,
#                     max_new_tokens=max_new_tokens, do_sample=do_sample))

## 4) Task A — Paraphrasing / Style
Implement `restyle(texto, ...)` using **system** + **user** messages:
- System: "Eres un asistente que reescribe en español sin añadir datos nuevos."
- User: Provide **tone** (e.g., *docente, claro y preciso*) and **target length** (e.g., *120-160 palabras*), plus the input `texto`.

Keep the Spanish literal below for testing.

In [ ]:
texto_es = (
    "Para elaborar una IPA casera, mantén el macerado a 66–67 °C durante 60 minutos, "
    "hierve 60 minutos, añade lúpulo en adiciones tardías y realiza dry hop en fermentación. "
    "Controla la temperatura de fermentación según la levadura ale (18–20 °C) y asegúrate de una buena sanitización."
)

# TODO: implement restyle() calling your generate_chat()
def restyle(texto, tono="docente, claro y preciso", longitud="120-160 palabras",
            temperature=0.4, top_p=0.95, max_new_tokens=180, do_sample=True):
    """Build a system+user prompt and call generate_chat().
    - System should forbid adding new facts.
    - User should include tone and target length (keep Spanish literal values).
    """
    raise NotImplementedError("Complete restyle() by composing messages and calling generate_chat().")

# Example calls (uncomment after implementing restyle())
# print('--- Original ---')
# print(texto_es)
# print('\n--- Reescritura (T=0.4) ---')
# print(restyle(texto_es, temperature=0.4))
# print('\n--- Reescritura (T=0.8) ---')
# print(restyle(texto_es, temperature=0.8))

## 5) Task B — Sentiment classification by prompting
We evaluate **zero-shot** and **few-shot** on a small synthetic dataset (60 items: pos/neg/neu). Implement `classify_sentiment(text, mode)`.

- System: *"Eres un clasificador de sentimiento en español. Devuelve solo: pos, neg o neu."*
- Zero-shot user: ask to classify one text.
- Few-shot user: include **three** examples (pos/neu/neg) **in Spanish** and then classify one text.
- Normalize model output to exactly one of: `pos`, `neg`, `neu`.

Run the metrics and confusion matrix after implementing the function.

In [ ]:
import random
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

random.seed(42)

# Spanish literals — keep them
positivas = [
  "Aroma cítrico intenso y espuma persistente.",
  "Fermentación limpia, sin off-flavors.",
  "Buen amargor y final seco, muy bebible.",
  "Carbonatación justa y color dorado brillante.",
  "Dry hop muy expresivo, excelente NEIPA.",
  "Atenuación perfecta, densidad final en rango."
]
negativas = [
  "Sabor a cartón húmedo (oxidación) evidente.",
  "Fenoles medicinales, probablemente contaminación.",
  "Diacetilo notable, a mantequilla.",
  "Subcarbonatada y turbia sin intención.",
  "Amargor áspero y desequilibrado.",
  "Aromas sulfurosos, levadura estresada."
]
neutras = [
  "Maceré a 66 °C durante una hora.",
  "La densidad original fue 1.060.",
  "Añadí 50 g de lúpulo en dry hop.",
  "Fermenté 10 días a 19 °C.",
  "Usé levadura US-05 en sobre.",
  "Carbonaté a 2.2 volúmenes de CO₂."
]

data = [(t,'pos') for t in positivas] + [(t,'neg') for t in negativas] + [(t,'neu') for t in neutras]
data = data*2  # 60 items (balanced)
random.shuffle(data)

# TODO: implement classify_sentiment() using your generate_chat()
def classify_sentiment(text, mode='zero'):
    """Return one of: 'pos', 'neg', 'neu'.
    - Build system+user messages.
    - For few-shot, include 3 Spanish examples (pos/neu/neg) before the target text. For example:
            'Texto: "Aroma fresco, fermentación limpia y final seco."\nEtiqueta: pos\n'
            'Texto: "Correcta para una primera cocción casera."\nEtiqueta: neu\n'
            'Texto: "Presenta sabores a mantequilla y solvente."\nEtiqueta: neg\n'
    - Use greedy for deterministic eval: temperature=0.0, do_sample=False.
    """
    raise NotImplementedError("Complete classify_sentiment() for zero-shot and few-shot.")

def run_eval(mode='zero'):
    y_true, y_pred = [], []
    for text, lab in data:
        y_true.append(lab)
        y_pred.append(classify_sentiment(text, mode=mode))
    print('\n== Results', mode.upper(), '==')
    print(classification_report(y_true, y_pred, digits=3))
    cm = confusion_matrix(y_true, y_pred, labels=['pos','neg','neu'])
    print('Confusion matrix (pos/neg/neu):\n', cm)
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(f'Confusion matrix — {mode}')
    plt.colorbar()
    ticks = ['pos','neg','neu']
    plt.xticks(range(3), ticks)
    plt.yticks(range(3), ticks)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    return cm

# Uncomment after implementing classify_sentiment()
# cm_zero = run_eval('zero')
# cm_few  = run_eval('few')

## 6) Decoding experiment
Generate multiple paraphrases while changing `temperature` and `top_p`. Comment on **diversity vs coherence**.

**Important**: If `temperature=0.0`, use **greedy** (`do_sample=False`).

In [ ]:
settings = [
    (0.0, 1.0),
    (0.2, 0.95),
    (0.8, 0.90),
]

# Uncomment after implementing restyle()
# for t, p in settings:
#     print(f"\n### temperature={t}, top_p={p}")
#     out = restyle(texto_es, temperature=t, top_p=p, do_sample=(t>0.0))
#     print(out)


## 7) Playground — **Multi‑turn conversation**
Implement a minimal multi-turn loop with persistent history. Keep Spanish literals in prompts.

**Implement:**
- `reset_chat(system=...)` → initialize history with a system prompt.
- `chat_turn(user_text, ...)` → append user message, call `generate_chat(history, ...)`, append assistant reply, return text.

Tip: print a compact view of the history when `show_history=True`.

In [ ]:
CHAT_HISTORY = []

# TODO: implement reset_chat() and chat_turn()
def reset_chat(system="Eres maestro cervecero. Responde en español con pasos claros y consejos de seguridad alimentaria."):
    """Start the history with a system message."""
    raise NotImplementedError("Implement reset_chat() to initialize CHAT_HISTORY.")

def chat_turn(user_text, temperature=0.2, top_p=0.95, max_new_tokens=256, do_sample=True, show_history=False):
    """Append user turn, generate assistant reply with generate_chat(), store it, and return the reply."""
    raise NotImplementedError("Implement chat_turn() using generate_chat() and CHAT_HISTORY.")

# Example usage (uncomment after implementing)
# reset_chat()
# _ = chat_turn("Quiero hacer una IPA casera de 20 litros: ¿qué perfil de macerado y lúpulos recomiendas?", temperature=0.3, do_sample=True)
# _ = chat_turn("¿Cómo controlo la fermentación y evito off-flavors como diacetilo u oxidación?", temperature=0.3, do_sample=True, show_history=True)

## 8) Mini BJCP table + prompt helper
Use the mini BJCP table (read-only) and implement a function that builds a **recipe/process prompt** for a selected style. Keep literals (style ranges) in Spanish.

**Implement:**
- `style_prompt(estilo, ...)` → returns a `messages` list for `generate_chat()` using BJCP ranges.

Then, call `generate_chat(messages, ...)` to get a recipe/process suggestion in Spanish.

In [ ]:
import pandas as pd

BJCP_MINI = pd.DataFrame([
    {
        'Estilo': 'American IPA', 'BJCP': '21A',
        'OG': '1.056–1.070', 'FG': '1.008–1.014', 'ABV': '5.5–7.5%', 'IBU': '40–70', 'SRM': '6–14',
        'Levadura': 'US-05 / WLP001 (Chico)', 'Lúpulos': 'Citra, Simcoe, Cascade',
        'Notas': 'Aromas cítricos/resinosos; amargor firme; final relativamente seco.'
    },
    {
        'Estilo': 'Oatmeal Stout', 'BJCP': '16B',
        'OG': '1.045–1.065', 'FG': '1.010–1.018', 'ABV': '4.2–5.9%', 'IBU': '25–40', 'SRM': '22–40',
        'Levadura': 'S-04 / Wyeast 1084 (Irish Ale)', 'Lúpulos': 'East Kent Goldings, Fuggles',
        'Notas': 'Cuerpo cremoso, avena; tostados/chocolate; amargor moderado.'
    },
    {
        'Estilo': 'German Pils', 'BJCP': '5D',
        'OG': '1.044–1.050', 'FG': '1.008–1.013', 'ABV': '4.4–5.2%', 'IBU': '22–40', 'SRM': '2–5',
        'Levadura': 'W-34/70 / WLP830 (German Lager)', 'Lúpulos': 'Hallertau, Tettnang, Saaz',
        'Notas': 'Amargor limpio y definido; final seco; alta atenuación; lagering frío.'
    }
])

def show_bjcp():
    display(BJCP_MINI)

# TODO: implement style_prompt() that returns (messages, info_dict)
def style_prompt(estilo,
                 volumen_l=20,
                 macerado="66–67 °C 60 min",
                 hervido="60 min",
                 objetivo_abv=None,
                 objetivo_ibu=None,
                 dry_hop=True,
                 restricciones="No inventes datos; prioriza sanitización y evita off-flavors (oxidación, diacetilo).",
                 salida="pasos + lista de ingredientes en gramos, cronograma de lúpulos y control de fermentación"):
    """Return a system+user messages list using BJCP ranges and the chosen style.
    - Look up the row by 'Estilo'.
    - Compose a Spanish prompt with ranges and process constraints.
    - Return (messages, info_dict) where info_dict is the row as a Python dict.
    """
    raise NotImplementedError("Complete style_prompt() to build messages for generate_chat().")

# Example (uncomment after implementation)
# show_bjcp()
# msgs, info = style_prompt('American IPA', volumen_l=20, objetivo_abv='6.5%', objetivo_ibu='55–60', dry_hop=True)
# print("\n--- Prompt preview (first 300 chars) ---")
# print(msgs[1]['content'][:300] + '...')
# print("\n--- Model reply (first 600 chars) ---")
# print(generate_chat(msgs, temperature=0.3, top_p=0.95, max_new_tokens=400)[:600])